# Provenance-verification sensitivity analysis (ZTAS-DTh revision, R1)

**Important:** `p_v` is a *simulation parameter*: the assumed probability that a phase-boundary provenance gate rejects a malicious artifact. It is **not** a measured detection rate. `f_art` is the assumed share of flows on which lateral movement is artifact-mediated (only those moves can be stopped by a provenance gate). v1 implicitly used `f_art = 1`.

Inputs: `../data/raw/e1_pv.csv` (generated by `code/run_e1.py pv`), summarised by `code/analyze_pv.py` into `../data/processed/provenance_sensitivity.csv` (hierarchical-bootstrap 95% CIs; 30 graphs x 20 runs per cell).

In [1]:
import pandas as pd, numpy as np, matplotlib
matplotlib.use('Agg'); import matplotlib.pyplot as plt
ps = pd.read_csv('../data/processed/provenance_sensitivity.csv')
ps.shape, ps.topology.unique()

((720, 22),
 <StringArray>
 [            'dense',     'feedback_loop', 'high_interconnect',
          'hub_spoke',         'multi_org',    'multi_supplier',
          'reference',       'shared_repo',            'sparse']
 Length: 9, dtype: str)

## 1. Required p_v grid (0.1, 0.3, 0.5, 0.7, 0.9, 0.99), reference topology

In [2]:
grid = [0.1,0.3,0.5,0.7,0.9,0.99]
cols = ['profile','config','f_art','p_v','compromised_fraction','propagation_prob_to_operations','containment_prob','detection_prob','residual_compromise']
ref = ps[(ps.topology=='reference') & ps.p_v.isin(grid)][cols].sort_values(['f_art','config','profile','p_v'])
ref.round(3)

,profile,config,f_art,p_v,compromised_fraction,propagation_prob_to_operations,containment_prob,detection_prob,residual_compromise
483,noisy,zt_prov,0.5,0.10,0.775,0.802,0.198,0.998,0.098
484,noisy,zt_prov,0.5,0.30,0.700,0.722,0.278,0.998,0.094
485,noisy,zt_prov,0.5,0.50,0.615,0.610,0.390,0.998,0.088
486,noisy,zt_prov,0.5,0.70,0.548,0.533,0.465,0.998,0.081
488,noisy,zt_prov,0.5,0.90,0.486,0.453,0.540,0.998,0.077
490,noisy,zt_prov,0.5,0.99,0.461,0.420,0.573,0.998,0.074
523,stealthy,zt_prov,0.5,0.10,0.775,0.802,0.198,0.998,0.098
524,stealthy,zt_prov,0.5,0.30,0.700,0.722,0.278,0.998,0.094
525,stealthy,zt_prov,0.5,0.50,0.615,0.610,0.390,0.998,0.088
526,stealthy,zt_prov,0.5,0.70,0.548,0.533,0.465,0.998,0.081


## 2. Difference between attacker profiles (stealthy - noisy), reference topology

In [3]:
r = ps[(ps.topology=='reference') & ps.p_v.isin(grid) & (ps.config=='ztas')]
piv = r.pivot_table(index=['f_art','p_v'], columns='profile', values='compromised_fraction')
piv['stealthy_minus_noisy'] = piv['stealthy'] - piv['noisy']; piv.round(3)

profile     noisy  stealthy  stealthy_minus_noisy
f_art p_v                                        
0.5   0.10  0.028     0.212                 0.184
      0.30  0.028     0.183                 0.155
      0.50  0.027     0.156                 0.128
      0.70  0.027     0.138                 0.111
      0.90  0.027     0.126                 0.099
      0.99  0.026     0.119                 0.092
1.0   0.10  0.028     0.193                 0.165
      0.30  0.027     0.140                 0.113
      0.50  0.026     0.102                 0.076
      0.70  0.025     0.078                 0.053
      0.90  0.025     0.067                 0.042
      0.99  0.025     0.063                 0.039

## 3. Sanity check: ZTAS-DTh at p_v = 0 must equal the 'no provenance' ablation (same paired draws)

In [4]:
a = ps[(ps.config=='ztas') & (ps.p_v==0.0)].set_index(['topology','profile','f_art'])['compromised_fraction']
b = ps[(ps.config=='ztas_no_prov')].set_index(['topology','profile','f_art'])['compromised_fraction']
print('max |difference| =', float((a-b).abs().max()))

max |difference| = 0.0


## 4. Across topologies (f_art = 0.5, stealthy attacker)

In [5]:
t = ps[(ps.f_art==0.5) & (ps.profile=='stealthy') & ps.config.isin(['zt_prov','ztas']) & ps.p_v.isin(grid)]
t.pivot_table(index=['topology','config'], columns='p_v', values='compromised_fraction').round(3)

p_v                         0.10   0.30   0.50   0.70   0.90   0.99
topology          config                                           
dense             zt_prov  0.903  0.877  0.832  0.785  0.739  0.718
                  ztas     0.582  0.497  0.406  0.348  0.284  0.265
feedback_loop     zt_prov  0.821  0.706  0.617  0.534  0.461  0.425
                  ztas     0.305  0.225  0.174  0.146  0.120  0.112
high_interconnect zt_prov  0.869  0.792  0.723  0.659  0.597  0.567
                  ztas     0.452  0.341  0.259  0.211  0.181  0.160
hub_spoke         zt_prov  0.487  0.361  0.292  0.245  0.215  0.206
                  ztas     0.098  0.083  0.076  0.069  0.064  0.063
multi_org         zt_prov  0.706  0.447  0.276  0.180  0.124  0.109
                  ztas     0.201  0.126  0.089  0.070  0.058  0.052
multi_supplier    zt_prov  0.789  0.668  0.588  0.505  0.449  0.427
                  ztas     0.219  0.164  0.138  0.117  0.105  0.102
reference         zt_prov  0.775  0.700  0.615  0.548  0.486  0.461
                  ztas     0.212  0.183  0.156  0.138  0.126  0.119
shared_repo       zt_prov  0.802  0.638  0.484  0.388  0.316  0.286
                  ztas     0.302  0.194  0.136  0.110  0.100  0.093
sparse            zt_prov  0.121  0.095  0.084  0.075  0.072  0.071
                  ztas     0.045  0.042  0.040  0.037  0.036  0.036

In [6]:
fig, ax = plt.subplots(1, 2, figsize=(9, 3.2))
for i, prof in enumerate(['noisy','stealthy']):
    for cfg, c in [('zt_prov','#eb6834'),('ztas','#2a78d6')]:
        for fa, ls in [(0.5,'-'),(1.0,'--')]:
            d = ps[(ps.topology=='reference')&(ps.profile==prof)&(ps.config==cfg)&(ps.f_art==fa)].sort_values('p_v')
            ax[i].plot(d.p_v, d.compromised_fraction, ls, color=c, marker='o', ms=3, label=f'{cfg}, f_art={fa}')
            ax[i].fill_between(d.p_v, d.compromised_fraction_ci_lo, d.compromised_fraction_ci_hi, color=c, alpha=0.12)
    ax[i].set_title(f'{prof} attacker (reference topology)'); ax[i].set_xlabel('p_v (assumed)'); ax[i].set_ylim(0,1)
ax[0].set_ylabel('compromised fraction'); ax[1].legend(fontsize=7)
fig.tight_layout(); fig.savefig('../figures/nb_provenance_sensitivity.png', dpi=150)

## 5. Interpretation (see PROVENANCE_SENSITIVITY_REPORT.md)
* Provenance verification matters most in the **static ZT + provenance** baseline: its compromised fraction falls steeply with p_v, and far more steeply when all propagation is artifact-mediated (f_art = 1, the v1 assumption).
* For **full ZTAS-DTh** under f_art = 0.5 the dependence on p_v is moderate (stealthy: about 0.24 at p_v = 0 to 0.12 at p_v = 0.99); segmentation and the risk engine carry much of the effect. For the noisy attacker the risk engine alone contains the attack and p_v barely matters.
* Every number here is conditional on assumed parameters; no value of p_v has been measured.